# TradeOps Ghana — Trade Intelligence Analysis
## Methodology, Data Sources & Verification Notebook

**Purpose:** This notebook walks through every step of the TradeOps analysis so the methodology can be verified and audited.

**Business context:** Finding viable import/export commodity opportunities for a Ghana-based trader with GHC 100,000 (~$6,500 USD) starting capital.

**Reference:** Kenneth D. Weiss — *Building an Import/Export Business* (2007, Wiley)  
**Data sources:** World Bank API, UN Comtrade, FAOSTAT, Ghana Statistical Service research

---
### Methodology Overview
1. Load Ghana macro trade data (World Bank API)
2. Load commodity-level trade flows (UN Comtrade / research)
3. Load commodity price trends (FAOSTAT / World Bank)
4. Apply opportunity scoring matrix (5 factors, Weiss framework)
5. Generate ranked opportunities and supply-demand graph
6. Validate results against published Ghana trade statistics

In [ ]:
# Setup — add scripts directory to path
import sys
import json
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'scripts'))

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
plotly_config = {'displayModeBar': False}

print(f'Repo root: {REPO_ROOT}')
print('Setup complete.')

## Step 1 — Ghana Macro Trade Data

**Source:** World Bank API (api.worldbank.org/v2/country/GHA/)

We fetch six key macroeconomic indicators for Ghana covering the most recent 6 years. The World Bank API is open and requires no authentication.

In [ ]:
from fetch_ghana import fetch_ghana_data, get_latest_ghana_data, INDICATORS, SEED_DATA

print('World Bank Indicators we fetch:')
for key, code in INDICATORS.items():
    print(f'  {key:25s} → {code}')

# Load the most recent data (from cache or live fetch)
ghana_data = get_latest_ghana_data()
print(f'\nData loaded. Keys: {list(ghana_data.keys())}')

In [ ]:
# Inspect Ghana macro time-series
macro_ts = ghana_data.get('macro_timeseries', {})

if macro_ts:
    years = sorted(set(y for v in macro_ts.values() for y in v.keys()), reverse=True)
    rows = []
    for yr in years:
        row = {'Year': yr}
        for k, v in macro_ts.items():
            row[k] = v.get(yr)
        rows.append(row)
    df_macro = pd.DataFrame(rows)
    print('World Bank Live Data (most recent years):')
    display(df_macro.head(6))
else:
    print('No live World Bank time-series available (using embedded seed data).')

# Embedded seed data for trade balance
print('\nEmbedded Research Data (2021-2023):')
display(pd.DataFrame(ghana_data.get('macro', SEED_DATA.get('macro', {}))).T)

In [ ]:
# Visualise Ghana trade balance
years = [2021, 2022, 2023]
imports_b = [13.9, 17.2, 16.4]
exports_b = [15.3, 16.1, 16.9]
balance = [e - i for e, i in zip(exports_b, imports_b)]

fig = go.Figure()
fig.add_trace(go.Bar(name='Exports ($B)', x=years, y=exports_b, marker_color='#2ecc71'))
fig.add_trace(go.Bar(name='Imports ($B)', x=years, y=imports_b, marker_color='#e74c3c'))
fig.add_trace(go.Scatter(name='Balance ($B)', x=years, y=balance, mode='lines+markers',
                          line=dict(color='#f39c12', width=3)))
fig.update_layout(barmode='group', title='Ghana Merchandise Trade Balance 2021-2023',
                   yaxis_title='USD Billions')
fig.show(config=plotly_config)

## Step 2 — Top Import & Export Commodities

**Sources:**
- Ghana Statistical Service (StatsBank) published trade reports
- World Integrated Trade Solution (WITS) — Ghana country snapshot
- Published media: GhanaWeb, Graphic Online (2024 trade reports)

Data is embedded as research-validated seed values. UN Comtrade API can supplement with commodity-level HS code data when an API key is available.

In [ ]:
# Top Ghana Imports
top_imports = ghana_data.get('top_imports', [])
df_imp = pd.DataFrame(top_imports)
df_imp['Rank'] = range(1, len(df_imp) + 1)
df_imp = df_imp[['Rank', 'commodity', 'value_usd_m', 'share_pct', 'yoy_growth', 'hs']]
df_imp.columns = ['Rank', 'Commodity', 'Value ($M)', 'Share (%)', 'YoY Growth (%)', 'HS Code']

print('Top Ghana Imports by Value (2023):')
display(df_imp.style.background_gradient(subset=['Value ($M)'], cmap='Reds')
              .format({'Value ($M)': '${:,.0f}M', 'YoY Growth (%)': '{:+.1f}%'}))

In [ ]:
# Top Ghana Non-Traditional Exports
top_exports = ghana_data.get('top_exports_nontrad', [])
df_exp = pd.DataFrame(top_exports)
df_exp['Rank'] = range(1, len(df_exp) + 1)
df_exp = df_exp[['Rank', 'commodity', 'value_usd_m', 'share_pct', 'yoy_growth', 'hs']]
df_exp.columns = ['Rank', 'Commodity', 'Value ($M)', 'Share (%)', 'YoY Growth (%)', 'HS Code']

print('Top Ghana Non-Traditional Exports by Value (2023):')
display(df_exp.style.background_gradient(subset=['YoY Growth (%)'], cmap='Greens')
              .format({'Value ($M)': '${:,.0f}M', 'YoY Growth (%)': '{:+.1f}%'}))

## Step 3 — Commodity Price Analysis

**Source:** FAOSTAT (fenixservices.fao.org) producer price database + World Bank Pink Sheet research

We track prices over 5 years (2019-2023) for both export commodities (Ghana producer price vs. world buyer price) and import commodities (import price vs. Ghana retail price). The margin potential is the key metric.

In [ ]:
from fetch_prices import get_latest_prices_data

prices_data = get_latest_prices_data()
prices = prices_data.get('prices', {})

# Build summary table
rows = []
for comm, pdata in prices.items():
    latest_price = pdata.get('trend', [{}])[-1].get('price_usd_mt') if pdata.get('trend') else None
    rows.append({
        'Commodity': comm,
        'Ghana Price ($/MT)': pdata.get('ghana_producer_usd_mt') or pdata.get('import_price_usd_mt'),
        'World/Retail Price ($/MT)': pdata.get('world_avg_usd_mt') or pdata.get('ghana_retail_usd_mt'),
        'Margin Potential (%)': pdata.get('margin_potential_pct'),
        'FAOSTAT Latest Price': latest_price,
    })

df_prices = pd.DataFrame(rows)
print('Commodity Price Summary:')
display(df_prices.style
        .background_gradient(subset=['Margin Potential (%)'], cmap='RdYlGn')
        .format({'Margin Potential (%)': '{:.0f}%'}, na_rep='N/A'))

In [ ]:
# Price trend chart
PRICE_DATA = {
    'Ginger':         [580, 640, 710, 780, 820],
    'Shea butter':    [450, 520, 610, 690, 750],
    'Cashew nuts':    [850, 920, 980, 1050, 1100],
    'Moringa powder': [1400, 1700, 1900, 2050, 2200],
    'Rice (import)':  [390, 400, 430, 490, 620],
    'Poultry (import)':[1400, 1500, 1620, 1750, 1811],
}
years = [2019, 2020, 2021, 2022, 2023]

fig = go.Figure()
for comm, vals in PRICE_DATA.items():
    fig.add_trace(go.Scatter(x=years, y=vals, mode='lines+markers', name=comm))
fig.update_layout(title='Commodity Price Trends 2019-2023 (USD/MT)',
                   yaxis_title='Price (USD/MT)', xaxis_title='Year')
fig.show(config=plotly_config)

## Step 4 — Opportunity Scoring Methodology

Each commodity is scored on 5 factors (total 100 points):

| Factor | Weight | How Measured |
|---|---|---|
| Gap/surplus size | 30% | Ghana import/export value vs. reference max |
| YoY growth trend | 25% | Year-over-year % growth in trade volume |
| Gross margin potential | 25% | (sell price - buy price) / buy price; threshold = 20% min |
| Logistics feasibility | 10% | Distance to supplier/market (near/medium/far) |
| Budget fit | 10% | Can GHC 60,000 order buy ≥5 MT (import) or ≥2 MT (export)? |

**Minimum viable margin (Weiss rule):** Any opportunity scoring <20% gross margin is flagged RED and excluded from recommendations.

In [ ]:
from fetch_comtrade import get_latest_comtrade_data
from analyze_gaps import run_analysis

comtrade_data = get_latest_comtrade_data()
prices_data = get_latest_prices_data()

analysis = run_analysis(ghana_data, comtrade_data, prices_data)
print(f"Analysis complete.")
print(f"  Import opportunities found: {analysis['summary']['total_import_opps']}")
print(f"  Export opportunities found: {analysis['summary']['total_export_opps']}")
print(f"  Green margin opportunities: {analysis['summary']['green_margin_opps']}")
print(f"  Budget-fit opportunities: {analysis['summary']['budget_fit_opps']}")

In [ ]:
# Import opportunity scores
import_opps = analysis['import_opportunities']
df_imp_opps = pd.DataFrame([{
    'Commodity': o['commodity'],
    'Score': o['score'],
    'Import $M': o['ghana_import_usd_m'],
    'YoY Growth %': o['yoy_growth_pct'],
    'Best Supplier': o['best_supplier'],
    'Price $/MT': o['best_price_usd_mt'],
    'Margin %': o['margin_pct'],
    'Budget Fit': 'Yes' if o['budget_fit'] else 'Tight',
    'Order Qty (MT)': o['est_order_qty_mt'],
} for o in import_opps])

print('Import Opportunity Scores:')
display(df_imp_opps.style
        .background_gradient(subset=['Score'], cmap='RdYlGn', vmin=0, vmax=100)
        .format({'Margin %': '{:.0f}%', 'Score': '{:.0f}'})
        .set_caption('Scored against 5-factor matrix. Min score 70 = Strong Opportunity.'))

In [ ]:
# Export opportunity scores
export_opps = analysis['export_opportunities']
df_exp_opps = pd.DataFrame([{
    'Commodity': o['commodity'],
    'Score': o['score'],
    'Export $M': o['ghana_export_usd_m'],
    'YoY Growth %': o['yoy_growth_pct'],
    'Best Market': o['best_market'],
    'Market Demand Growth %': o['market_demand_growth_pct'],
    'Ghana Price $/MT': o['ghana_producer_price_usd_mt'],
    'Margin %': o['margin_pct'],
    'Budget Fit': 'Yes' if o['budget_fit'] else 'Tight',
} for o in export_opps])

print('Export Opportunity Scores:')
display(df_exp_opps.style
        .background_gradient(subset=['Score'], cmap='RdYlGn', vmin=0, vmax=100)
        .format({'Margin %': '{:.0f}%', 'Score': '{:.0f}'})
        .set_caption('Shea butter leads with 84/100 — 116% YoY growth and 45% margin.'))

In [ ]:
# Score breakdown visualisation for top 3
import plotly.graph_objects as go
from plotly.subplots import make_subplots

top3 = sorted(import_opps + export_opps, key=lambda x: x['score'], reverse=True)[:3]
factor_names = list(top3[0]['score_breakdown'].keys())

fig = make_subplots(rows=1, cols=3, specs=[[{'type': 'polar'}]*3],
                    subplot_titles=[o['commodity'] for o in top3])

max_vals = {'gap_size': 30, 'growth': 25, 'margin': 25, 'logistics': 10,
            'budget_fit': 10, 'export_value': 30, 'market_demand': 10}

for i, opp in enumerate(top3, 1):
    breakdown = opp['score_breakdown']
    categories = list(breakdown.keys())
    pct_vals = [round(v / max_vals.get(k, 10) * 100) for k, v in breakdown.items()]
    fig.add_trace(go.Scatterpolar(
        r=pct_vals + [pct_vals[0]],
        theta=categories + [categories[0]],
        fill='toself', name=opp['commodity']
    ), row=1, col=i)

fig.update_layout(title='Score Breakdown: Top 3 Opportunities (% of max per factor)', height=400)
fig.show(config=plotly_config)

## Step 5 — Supply-Demand Network Analysis

The network graph maps each commodity gap to its global supply or demand sources. Nodes are Ghana (orange), supplier countries (blue), and buyer markets (green). Edge labels show price or demand growth.

In [ ]:
import networkx as nx

graph_data = analysis['supply_demand_graph']
G = nx.DiGraph()
for node in graph_data['nodes']:
    G.add_node(node['id'], node_type=node['type'])
for edge in graph_data['edges']:
    G.add_edge(edge['from'], edge['to'], label=edge.get('label', ''), edge_type=edge.get('type', ''))

print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'\nGhana nodes (commodity gaps):')
for n, d in G.nodes(data=True):
    if d.get('node_type') == 'ghana':
        in_deg = G.in_degree(n)   # suppliers pointing in
        out_deg = G.out_degree(n) # buyers pointing out
        print(f'  {n}: {in_deg} suppliers / {out_deg} buyers')

In [ ]:
# Visualise the network
pos = nx.spring_layout(G, seed=42, k=2.5)
type_colors = {'ghana': '#f39c12', 'supplier': '#3498db', 'buyer': '#2ecc71'}

edge_x, edge_y = [], []
for u, v in G.edges():
    x0, y0 = pos[u]; x1, y1 = pos[v]
    edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

node_x = [pos[n][0] for n in G.nodes()]
node_y = [pos[n][1] for n in G.nodes()]
node_text = list(G.nodes())
node_colors = [type_colors.get(G.nodes[n].get('node_type', 'buyer'), '#95a5a6') for n in G.nodes()]
node_sizes = [30 if G.nodes[n].get('node_type') == 'ghana' else 18 for n in G.nodes()]

fig = go.Figure()
fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines',
                          line=dict(color='#bdc3c7', width=1), hoverinfo='none'))
fig.add_trace(go.Scatter(x=node_x, y=node_y, mode='markers+text',
                          text=node_text, textposition='top center',
                          marker=dict(color=node_colors, size=node_sizes),
                          hoverinfo='text'))
fig.update_layout(title='Ghana Trade Opportunity Network',
                   xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                   yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                   height=550, showlegend=False)
fig.show(config=plotly_config)

## Step 6 — Top 5 Opportunities: Final Conclusions

Final ranked list after applying all scoring factors. These are our recommendations for the GHC 100,000 budget.

In [ ]:
top5 = analysis['top_5_overall']

for rank, opp in enumerate(top5, 1):
    direction = 'IMPORT' if opp['type'] == 'import_opportunity' else 'EXPORT'
    margin_status = 'GREEN' if opp['margin_pct'] >= 30 else ('YELLOW' if opp['margin_pct'] >= 20 else 'RED')
    print(f"\n{'='*55}")
    print(f"#{rank} {opp['commodity']} ({direction}) — Score: {opp['score']}/100")
    print(f"   Margin: {opp['margin_pct']}% [{margin_status}]")
    if opp['type'] == 'import_opportunity':
        print(f"   Ghana imports: ${opp['ghana_import_usd_m']}M/yr (+{opp['yoy_growth_pct']}% YoY)")
        print(f"   Best source: {opp['best_supplier']} @ ${opp['best_price_usd_mt']}/MT")
    else:
        print(f"   Ghana exports: ${opp['ghana_export_usd_m']}M/yr (+{opp['yoy_growth_pct']}% YoY)")
        print(f"   Best market: {opp['best_market']} (+{opp['market_demand_growth_pct']}% demand)")
    print(f"   Starting order: GHC {opp['est_order_ghc']:,} → {opp['est_order_qty_mt']} MT")

In [ ]:
# Final bar chart — all opportunities ranked
all_opps = sorted(import_opps + export_opps, key=lambda x: x['score'], reverse=True)
df_all = pd.DataFrame([{'Commodity': o['commodity'],
                         'Score': o['score'],
                         'Type': 'Import' if o['type'] == 'import_opportunity' else 'Export',
                         'Margin %': o['margin_pct']} for o in all_opps])

fig = px.bar(df_all, x='Commodity', y='Score', color='Type',
              color_discrete_map={'Import': '#3498db', 'Export': '#2ecc71'},
              title='All Opportunity Scores — Ghana Trade Intelligence (GHC 100k budget)',
              text='Score')
fig.add_hline(y=70, line_dash='dash', line_color='green', annotation_text='Strong (70+)')
fig.add_hline(y=50, line_dash='dash', line_color='orange', annotation_text='Moderate (50+)')
fig.update_layout(yaxis_title='Opportunity Score (/100)', xaxis_title='')
fig.show(config=plotly_config)

## Step 7 — Budget Allocation Validation

Verifying that the recommended starting order for the top opportunity (Shea butter) fits within the GHC 100,000 budget while maintaining a safe working capital buffer, as per Weiss's 90-day runway rule.

In [ ]:
budget_ghc = analysis['budget_ghc']
usd_ghc = 15.4

# Shea butter calculation (top opportunity)
shea = next(o for o in export_opps if o['commodity'] == 'Shea butter')
shea_price_usd_mt = shea['ghana_producer_price_usd_mt']
order_budget_ghc = 55_000
order_usd = order_budget_ghc / usd_ghc
order_qty_mt = order_usd / shea_price_usd_mt
margin = shea['margin_pct'] / 100
gross_revenue = order_usd * (1 + margin)
profit_usd = gross_revenue - order_usd
profit_ghc = profit_usd * usd_ghc

print('=== Shea Butter Export — Financial Validation ===')
print(f'Total budget:         GHC {budget_ghc:>10,}')
print(f'Sourcing (55%):       GHC {order_budget_ghc:>10,}')
print(f'Working capital:      GHC {25_000:>10,}')
print(f'Market research:      GHC {10_000:>10,}')
print(f'Customs/certs:        GHC {7_000:>10,}')
print(f'Emergency reserve:    GHC {3_000:>10,}')
print(f'TOTAL:                GHC {budget_ghc:>10,}')
print()
print(f'Order size:           {order_qty_mt:.1f} MT of shea butter')
print(f'Ghana producer price: ${shea_price_usd_mt}/MT')
print(f'Est. export value:    ${gross_revenue:,.0f}')
print(f'Gross margin:         {shea["margin_pct"]}%')
print(f'Est. profit:          GHC {profit_ghc:,.0f}')
print(f'\nWEISS CHECK: Gross margin {shea["margin_pct"]}% >= 20% minimum? {"PASS" if shea["margin_pct"] >= 20 else "FAIL"}')
print(f'WEISS CHECK: Working capital covers 3 months? {"PASS" if 25_000 >= 15_000 else "FAIL"}')

## Data Sources & Verification

| Source | Data Used | Verified Against |
|---|---|---|
| World Bank API | Ghana macro indicators (GDP, imports, exports) | World Bank Ghana page |
| WITS (World Bank) | Ghana trade snapshot 2023 ($16.4B imports, $16.9B exports) | GhanaWeb business reports |
| Ghana Statistical Service | Top commodity breakdowns | StatsBank.statsghana.gov.gh |
| FAOSTAT | Producer prices (when API available) | FAO data tables |
| UN Comtrade | Commodity-level partner flows | comtradeplus.un.org |
| GhanaWeb / Graphic Online | Non-traditional export rankings 2024 | Published 2024 trade articles |

**Limitations:**
- FAOSTAT API was unavailable during the run (timeout) — embedded 5-year price research used instead
- UN Comtrade live API requires a subscription key — embedded 2023 research data used
- Prices and volumes are 2023 annual data; may differ from current spot prices
- Exchange rate: GHC 15.4 / USD (approximate 2025 rate)

**Next steps to improve accuracy:**
1. Register at comtradeapi.un.org for a free API key, set as `COMTRADE_API_KEY` environment variable
2. Subscribe to Ghana Statistical Service data alerts at statsghana.gov.gh
3. Check World Bank Commodity Price Data (Pink Sheet) monthly for price updates